# Notebook 02: Feature Engineering

Generates synthetic delivery-zone features, builds spatial weights and lag variables, and assembles the feature matrix for downstream ML training.

## Imports and load grid

In [ ]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import numpy as np
import pandas as pd
import joblib
import os

gdf = gpd.read_file('../data/processed/grid.geojson')
print(f"Loaded grid with {len(gdf)} cells")
gdf.head()

## Generate synthetic feature data

Attaches `pop_density`, `income_index`, `competitor_count`, `road_density`, `transit_stops`, `warehouse_proximity_km`, and the binary `profitable` label.

In [ ]:
from src.feature_engineering import generate_synthetic_data, add_spatial_lag_features, build_feature_matrix

gdf = generate_synthetic_data(gdf)
print(f"Columns after synthetic data: {list(gdf.columns)}")
gdf.head()

## Build spatial weights and lag features

In [ ]:
from src.spatial_weights import build_weights_matrix, save_weights

weights = build_weights_matrix(gdf)

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

save_weights(weights, '../data/processed/weights_matrix.pkl')
save_weights(weights, '../models/weights_matrix.pkl')

gdf = add_spatial_lag_features(gdf, weights)
print(f"Columns after spatial lags: {list(gdf.columns)}")

## Build feature matrix

In [ ]:
X, y, feature_names = build_feature_matrix(gdf)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Feature names: {feature_names}")

## Save X_train and y_train

In [ ]:
joblib.dump(X, '../data/processed/X_train.pkl')
joblib.dump(y, '../data/processed/y_train.pkl')

# Also persist the features DataFrame for the API model_loader
features_df = gdf[['grid_id'] + feature_names].copy()
features_df.to_pickle('../data/processed/features.pkl')

# Re-save grid with all new columns so downstream notebooks get them
gdf.to_file('../data/processed/grid.geojson', driver='GeoJSON')

print("Saved X_train.pkl, y_train.pkl, features.pkl, and updated grid.geojson")

## Class distribution

In [ ]:
n_total = len(y)
n_pos = int(y.sum())
n_neg = n_total - n_pos
pct_pos = n_pos / n_total * 100
pct_neg = n_neg / n_total * 100

print(f"Class 0 (not profitable): {n_neg} ({pct_neg:.1f}%)")
print(f"Class 1 (profitable):     {n_pos} ({pct_pos:.1f}%)")
print(f"Imbalance ratio (neg/pos): {n_neg / max(n_pos, 1):.1f}:1")

## Exploratory plots

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of pop_density
axes[0].hist(gdf['pop_density'].values, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Population Density', fontsize=13)
axes[0].set_xlabel('pop_density')
axes[0].set_ylabel('Count')

# Scatter of lat/lon coloured by profitable
colors = ['#dd3333' if p == 0 else '#22bb66' for p in gdf['profitable'].values]
axes[1].scatter(
    gdf['centroid_lon'].values,
    gdf['centroid_lat'].values,
    c=colors,
    s=2,
    alpha=0.7,
)
axes[1].set_title('Grid Cells (green = profitable)', fontsize=13)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.show()